# Temperature dependence of Hodgkin & Huxley action potentials

![image](https://www.gardeningknowhow.com/wp-content/uploads/2021/05/chili-pepper-400x300.jpg)

Hodgkin and Huxley worked on squid, in a tank, at **6.3 degrees**. Your
own neurons run some thirty degrees warmer. Does that matter?

It matters a great deal, and this notebook is about exactly *what* changes
and what does not.

This notebook grades your answers for you, and **each student records at
their own temperature**.

## Step 1: Setup

In [ ]:
# Setup inline plotting
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# For Google Colab, this line installs NEURON
#!pip install neuron quantities

In [ ]:
# Fetch mechanisms
# Uncomment this line if on google colab
#!git clone https://github.com/ABL-Lab/NSC6084-A26.git

In [ ]:
# Compile the mechanisms
# Note: recompiled mechanisms will not take effect until neuron is imported or the jupyter kernel is restarted

# Uncomment this line if on google colab
#!nrnivmodl ./NSC6084-A26/Sept15/mechanisms
# Uncomment this line if running locally
!nrnivmodl mechanisms

In [ ]:
# We will let this library handle unit conversion for us
import quantities as pq
from quantities import um, nS, mV, cm, ms, nA, S, uF, Hz, degrees, s, MOhm

In [ ]:
# Import and initialize NEURON
import neuron
from neuron import h
h.load_file("stdrun.hoc")

In [ ]:
# Import other modules we need
import numpy as np

## Step 1b: Load your personal exercise parameters

`your_celsius` is the temperature your preparation is held at, and
`step_v` is the voltage the clamp steps to in all three questions.

In [ ]:
from obi_notebook import grading

assignment = grading.load()

soma_length  = assignment.params["soma_length_um"]
g_leak       = assignment.params["g_leak_nS"]
your_celsius = assignment.params["celsius"]     # your recording temperature
step_v       = assignment.params["step_v_mV"]   # the questions step here

print(f"Your soma length:      {soma_length} um")
print(f"Your leak conductance: {g_leak} nS")
print(f"Your temperature:      {your_celsius} degC")
print(f"Your step voltage:     {step_v} mV")
print(f"Exercises to submit:   {assignment.exercise_keys}")

## Step 2: Define the circuit
We will use a single compartment, called a "Section" (more on that in next lectures). <br>
It has a cylindrical geometry with length "L" and a diameter "diam", and a specific capacitance "cm" (capacitance per area) <br>
**Unit conversion is a common source of error, so we will be explicit with our units.** 

In [ ]:
soma = h.Section()

### Query NEURON for the expected units for soma.L & soma.diam

In [ ]:
[h.units(x) for x in ["L", "diam"]]

In [ ]:
# soma.L is YOUR personal value, loaded in Step 1b above.
soma.L = soma_length * um
soma.diam =  10 * um

In [ ]:
volume = soma(0.5).volume() * um**3

In [ ]:
area = soma(0.5).area() * um**2

In [ ]:
area

In [ ]:
volume

### Assign the membrane capacitance "everywhere"

In [ ]:
h.units("cm")  # Query the expected units

In [ ]:
specific_membrane_capacitance = 1 * uF/cm**2

In [ ]:
for sec in soma.wholetree():
    sec.cm = specific_membrane_capacitance #  specific membrane capacitance (micro Farads / cm^2)
    sec.Ra = 100

### Add the Hodgkin-Huxley conductances

In [ ]:
# This model includes the transient Na+, persistent K+ and the leak conductances
soma.insert("hh")

That's almost too easy!

### Parametize the leak conductance G = 1/R

In [ ]:
G = g_leak * nS  # R = 1/G in our RC circuit -- YOUR personal value, see Step 1b

In [ ]:
v_rest = -70*mV

In [ ]:
tau_m = (specific_membrane_capacitance * area / G).rescale(ms)

In [ ]:
tau_m = soma(0.5).cm / soma(0.5).hh.gl

In [ ]:
# Assign the leak conductance everywhere
for seg in soma:
    seg.hh.gl = (G/area).rescale(S/cm**2)  # Compute specific conductance, and rescale to units of 'S/cm2'
    seg.hh.el = -54.3

In [ ]:
# Read it back off the model, now that gl has actually been assigned.
tau_m = ((soma(0.5).cm * uF/cm**2) / (soma(0.5).hh.gl * S/cm**2)).rescale(ms)
tau_m

### Inspect our parameters

In [ ]:
soma.psection()

In [ ]:
soma.nseg

### Add a current injection

In [ ]:
stim = h.IClamp(soma(0.5))

In [ ]:
stim.delay = 200 * ms   # wait 200 ms, so we can see the resting state first
stim.dur = 20 * ms      # a brief pulse: we want one action potential
stim.amp = 0.02 * (soma_length / 10.0) * nA   # scaled to your soma area

## Step 3: Run the simulation

### Define recordings of simulation variables

In [ ]:
soma_v = h.Vector().record(soma(0.5)._ref_v)
t = h.Vector().record(h._ref_t)

In [ ]:
# Record hh gating variables
hh_vars = ['h', 'm', 'n', 'gna', 'gk']
hh_recordings = {}
for var in hh_vars:
    ref = getattr(soma(0.5).hh, "_ref_"+var )
    hh_recordings[var] = h.Vector().record(ref) 

In [ ]:
hh_recordings

### What temperature does to the action potential

Hodgkin and Huxley's own range: the squid axon was studied between about
3 and 20 degrees.

In [ ]:
temp_range = [3.6, 6.3, 9.8, 13.3, 20.2]

In [ ]:
def run_current_clamp(temp=6.3):
    """ Run the current-clamp pulse at this temperature; return (t, v). """
    h.celsius = temp
    h.finitialize( float(v_rest) )
    h.continuerun( float(1000 * ms) )
    return np.array(t), np.array(soma_v)

In [ ]:
plt.figure()
for temp in temp_range:
    a_t, v = run_current_clamp(temp)
    plt.plot(a_t, v, label='%.1f$^\circ$C' % temp)
plt.axis([202,212,-80,50])
plt.legend()
plt.xlabel("t (ms)")
plt.ylabel("v (mV)")

The action potential gets **faster and smaller** as the cell warms up. Push
the temperature far enough and it disappears altogether -- the classic **heat
block**. Try adding 25 or 30 degrees to `temp_range` and see.

But *why* does it shrink? Nothing in the plot tells us whether the channels
are opening less or simply opening and closing faster. To separate those we
need voltage clamp.

---

## Step 4: Pull it apart with voltage clamp

We now clamp the membrane and look at the two currents one at a time, at
different temperatures.

In [ ]:
# zero the current clamp to disable it
stim.amp = 0

### Add the voltage clamp & define the step

In [ ]:
vclamp = h.SEClamp(soma(0.5))

In [ ]:
T_STEP = 200 * ms        # the step begins here
vclamp.dur1 = T_STEP     # hold at rest until then
vclamp.dur2 = 700 * ms   # hold the step voltage for 700 ms
vclamp.amp1 = v_rest     # holding potential before the step
vclamp.amp2 = -40*mV     # the step voltage (overwritten by the runs below)
vclamp.amp3 = v_rest     # back to rest afterwards

In [ ]:
(1/G).rescale(MOhm) # What is Rin in MOhm?

In [ ]:
vclamp.rs = 0.01 * MOhm # The clamp series resistance should be < 1/100 Rin

### Record the voltage clamp current

In [ ]:
vclamp_i = h.Vector().record(vclamp._ref_i)

### Functions to run a clamp step

Note the name: this is **not** the same function as `run_current_clamp`
above. Keeping them separate matters -- reusing one name for both is an easy
way to break the earlier figure without noticing.

In [ ]:
SKIP_MS = 0.2   # step past the capacitive artifact at the step edge

def set_conductances(gnabar=0.12, gkbar=0.036):
    """ Switch the HH Na+ and K+ conductances on or off (units: S/cm2). """
    for seg in soma:
        seg.hh.gnabar = gnabar
        seg.hh.gkbar = gkbar

def run_voltage_clamp(step_voltage, temp=6.3):
    """ Step to step_voltage at this temperature.

    Returns (time_since_step, leak_subtracted_current), both arrays, with the
    capacitive artifact at the step edge already trimmed off. The leak is
    subtracted by measuring late in the step, where the Na+ current has fully
    inactivated and the K+ current has fully activated.
    """
    h.celsius = temp
    vclamp.amp2 = step_voltage
    h.finitialize( float(v_rest) )
    h.continuerun( float(1000 * ms) )
    a_t, i = np.array(t), np.array(vclamp_i)
    t0 = float(T_STEP)
    late = i[np.argmin(np.abs(a_t - (t0 + 190)))]
    window = (a_t >= t0 + SKIP_MS) & (a_t <= t0 + 190)
    return a_t[window] - t0, i[window] - late

### The sodium current at three temperatures

In [ ]:
set_conductances(gnabar=0.12, gkbar=0.0)  # Na+ only
plt.figure()
for temp in [6.3, 13.3, 20.2]:
    dt, di = run_voltage_clamp(step_v, temp)
    plt.plot(dt, di, lw=1.5, label='%.1f$^\circ$C' % temp)
plt.legend()
plt.xlabel("time since step [ms]", size=14)
plt.ylabel("$I_{Na}$ [nA]", size=14)
plt.xlim(0, 15)

### There is the answer

The three currents rise and inactivate at completely different speeds -- and
they all reach **the same peak**.

That is not a coincidence. In `hh.mod` the temperature factor
$q_{10}^{(T-6.3)/10}$ divides the **time constants** and nothing else; the
steady-state curves $m_\infty(V)$, $h_\infty(V)$, $n_\infty(V)$ have no
temperature term at all. Warming the cell does not change where the gates
end up, only how quickly they get there.

So why did the *action potential* shrink? Because the membrane capacitance
and the leak do **not** speed up. At high temperature the K+ current
activates almost as fast as the Na+ current, and starts repolarising the
cell before the spike has finished rising. The two currents overlap, and the
spike is cut short.

---

## Now it's your turn! Questions with answer feedback

Answer each question for **your** neuron, using the parameters printed in
Step 1b. Each `submit` call grades one answer and leaves a record in StudiUM;
you can re-run a cell to resubmit a better answer.

Answers are scored on relative error: within 5% earns full credit, fading to
zero at 20% off.

### Question 1 -- How fast does the Na+ current inactivate?

Step to **your** `step_v` at **your** temperature, with only the Na+
conductance switched on, and fit the decay of the current from its peak.
Report the time constant in **ms**.

`decay_tau` below does the fitting. It takes the logarithm of the current and
fits a straight line to it -- the slope of that line is $-1/\tau$.

In [ ]:
def decay_tau(dt, di):
    """ Time constant of an exponentially decaying current, in ms.

    Works for both currents, because `run_voltage_clamp` already subtracted the
    steady late value: -di starts large and decays to zero either way. For the
    Na+ current that is the current itself inactivating; for the K+ current it
    is how far the current still has to climb to reach its plateau.

    The fit uses the interval where -di has fallen to between 5% and 0.2% of its
    peak. By 5% the fast gate has finished -- m for the Na+ current, the n^4
    curvature for the K+ one -- and what is left really is a single exponential;
    below 0.2% there is no longer enough current to measure.

    Taking a fraction of the peak rather than a fixed number of ms is what lets
    one window serve everybody: tau varies more than tenfold across the class,
    so a window long enough for the coldest cell would run far past the point
    where the warmest one has decayed to nothing.
    """
    y = -di                                  # positive, and decaying to zero
    peak = np.argmax(y)
    fit = (dt >= dt[peak]) & (y <= 0.05 * y[peak]) & (y >= 0.002 * y[peak])
    slope, _ = np.polyfit(dt[fit], np.log(y[fit]), 1)
    return -1.0 / slope

In [ ]:
set_conductances(gnabar=0.12, gkbar=0.0)  # Na+ only
dt, di = run_voltage_clamp(step_v, your_celsius)

na_tau = None # fill in

print(f"At {your_celsius} degC the Na+ current inactivates with tau = {na_tau:.4f} ms")
assignment.submit(na_tau, "na_tau")

### Question 2 -- How fast does the K+ current activate?

Now the other current: Na+ off, K+ on, same step and same temperature. The
delayed rectifier does not inactivate -- it climbs to a plateau and stays
there -- so what decays here is not the current but **how far it still has to
go** to reach that plateau.

That is the same shape as before, so the same `decay_tau` measures it. Report
the time constant in **ms**.

In [ ]:
set_conductances(gnabar=0.0, gkbar=0.036)  # K+ only
# fill in

k_tau = None # fill in

print(f"At {your_celsius} degC the K+ current activates with tau = {k_tau:.4f} ms")
assignment.submit(k_tau, "k_tau")

### Question 3 -- Measure the $q_{10}$

Now repeat **either** measurement at 6.3 degrees -- the temperature Hodgkin
and Huxley actually recorded at -- and report how many times faster your own cell is. That is the $q_10$ factor refers to the quantity on the left hand side below:

$$ q_{10}^{\,(T - 6.3)/10} = \frac{\tau(6.3^\circ\mathrm{C})}{\tau(T)} $$

and can be computed using the ratio on the right hand side, so no units.

**Do it with both currents.** You should get the same number twice, to within
your fitting error -- a single $q_{10}$ factor scales *every* rate in the model.
That is a strong assumption about the underlying biophysics, and it is worth
knowing that it is an assumption.

In [ ]:
set_conductances(gnabar=0.12, gkbar=0.0)   # Na+ only, at Hodgkin & Huxley's own temperature
dt, di = run_voltage_clamp(step_v, 6.3)
na_tau_ref = decay_tau(dt, di)

set_conductances(gnabar=0.0, gkbar=0.036)  # K+ only, same temperature
dt, di = run_voltage_clamp(step_v, 6.3)
k_tau_ref = decay_tau(dt, di)

print(f"from the Na+ current: {na_tau_ref/na_tau:.4f}")
print(f"from the K+ current:  {k_tau_ref/k_tau:.4f}")

q10_factor = None # fill in
assignment.submit(q10_factor, "q10_factor")

Finally, check it against the formula. `hh.mod` uses $q_{10} = 3$, so the
factor should be $3^{(T-6.3)/10}$ -- about 3 for every 10 degrees. Compare
that with what you measured.

A $q_{10}$ of 3 is large. It is why cold-blooded animals get sluggish in the
cold, why hypothermia is protective during surgery, and why a fever of two
degrees is a much bigger perturbation to your neurons than it sounds.

In [ ]:
print(f"measured: {q10_factor:.4f}")
print(f"3^((T-6.3)/10) = {3.0**((your_celsius-6.3)/10):.4f}")